# Phase 1 Data Review

Read-only inspection notebook for Phase 1 artifacts. This notebook loads existing Parquet/JSON outputs, summarizes coverage and quality, and does not write or mutate pipeline artifacts.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "data").exists() else cwd.parent
assert (ROOT / "data").exists(), f"Could not locate repo root from {cwd}"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
ROOT

PosixPath('/Users/shicheny/Documents/GitHub/deep-rl-portfolio-optimization')

## Load Artifacts

In [2]:
paths = {
    "raw_prices": ROOT / "data/raw/prices_daily.parquet",
    "raw_macro": ROOT / "data/raw/macro_daily.parquet",
    "features": ROOT / "data/processed/features_daily.parquet",
    "global_features": ROOT / "data/processed/global_features_daily.parquet",
    "features_normalized": ROOT / "data/processed/features_normalized_daily.parquet",
    "global_features_normalized": ROOT / "data/processed/global_features_normalized_daily.parquet",
    "model_matrix": ROOT / "data/processed/model_matrix_daily.parquet",
    "feature_spec": ROOT / "artifacts/feature_specs/feature_spec_v1.json",
    "data_quality_report": ROOT / "artifacts/reports/data_quality_report_v1.json",
}

missing = [name for name, path in paths.items() if not path.exists()]
assert not missing, f"Missing artifacts: {missing}"

prices = pd.read_parquet(paths["raw_prices"])
macro = pd.read_parquet(paths["raw_macro"])
features = pd.read_parquet(paths["features"])
global_features = pd.read_parquet(paths["global_features"])
features_normalized = pd.read_parquet(paths["features_normalized"])
global_features_normalized = pd.read_parquet(paths["global_features_normalized"])
model_matrix = pd.read_parquet(paths["model_matrix"])

feature_spec = json.loads(paths["feature_spec"].read_text())
data_quality_report = json.loads(paths["data_quality_report"].read_text())

display(Markdown("Loaded all Phase 1 artifacts."))

Loaded all Phase 1 artifacts.

## Artifact Summary

In [3]:
def artifact_summary(name: str, frame: pd.DataFrame) -> dict[str, object]:
    numeric = frame.select_dtypes(include="number")
    summary = {
        "artifact": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "nan_count": int(frame.isna().sum().sum()),
        "inf_count": int(np.isinf(numeric).sum().sum()) if not numeric.empty else 0,
    }
    if "date" in frame.columns:
        dates = pd.to_datetime(frame["date"])
        summary["start_date"] = dates.min().date().isoformat()
        summary["end_date"] = dates.max().date().isoformat()
    return summary

artifact_frames = {
    "raw_prices": prices,
    "raw_macro": macro,
    "features": features,
    "global_features": global_features,
    "features_normalized": features_normalized,
    "global_features_normalized": global_features_normalized,
    "model_matrix": model_matrix,
}

display(pd.DataFrame([artifact_summary(name, frame) for name, frame in artifact_frames.items()]))

,artifact,rows,columns,nan_count,inf_count,start_date,end_date
0,raw_prices,68026,12,536,0,2007-01-03,2026-04-27
1,raw_macro,26787,5,1002,0,2007-01-01,2026-04-27
2,features,57442,25,0,0,2010-01-04,2026-04-27
3,global_features,4103,11,0,0,2010-01-04,2026-04-27
4,features_normalized,57442,25,0,0,2010-01-04,2026-04-27
5,global_features_normalized,4103,11,0,0,2010-01-04,2026-04-27
6,model_matrix,4103,333,0,0,2010-01-04,2026-04-27


## Quality Report and Feature Spec

In [4]:
display(pd.DataFrame([data_quality_report]).T.rename(columns={0: "value"}))

spec_summary = {
    "feature_version": feature_spec["feature_version"],
    "asset_count": len(feature_spec["asset_order"]),
    "per_asset_feature_count": len(feature_spec["per_asset_features"]),
    "global_feature_count": len(feature_spec["global_features"]),
    "current_weight_feature_count": len(feature_spec["current_weight_features"]),
    "observation_dim": feature_spec["observation_dim"],
}
display(pd.DataFrame([spec_summary]).T.rename(columns={0: "value"}))

,value
universe_name,liquid_global_etf_v1
feature_version,v1
n_assets,14
model_start_date,2010-01-01
train_end_date,2023-12-31
validation_start_date,2024-01-01
test_start_date,2025-01-01
nan_count_final,0
inf_count_final,0
normalization_fit_split,train


,value
feature_version,v1
asset_count,14
per_asset_feature_count,21
global_feature_count,8
current_weight_feature_count,14
observation_dim,316


## Raw Coverage

In [5]:
price_coverage = (
    prices.assign(date=pd.to_datetime(prices["date"]))
    .groupby("ticker")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        rows=("date", "size"),
        adj_close_missing=("adj_close", lambda values: int(values.isna().sum())),
        volume_missing=("volume", lambda values: int(values.isna().sum())),
    )
    .reset_index()
)
price_coverage["start_date"] = price_coverage["start_date"].dt.date.astype(str)
price_coverage["end_date"] = price_coverage["end_date"].dt.date.astype(str)
display(price_coverage)

macro_coverage = (
    macro.assign(date=pd.to_datetime(macro["date"]))
    .groupby("series_id")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        rows=("date", "size"),
        value_missing=("value", lambda values: int(values.isna().sum())),
    )
    .reset_index()
)
macro_coverage["start_date"] = macro_coverage["start_date"].dt.date.astype(str)
macro_coverage["end_date"] = macro_coverage["end_date"].dt.date.astype(str)
display(macro_coverage)

,ticker,start_date,end_date,rows,adj_close_missing,volume_missing
0,DBC,2007-01-03,2026-04-27,4859,0,0
1,EEM,2007-01-03,2026-04-27,4859,0,0
2,EFA,2007-01-03,2026-04-27,4859,0,0
3,GLD,2007-01-03,2026-04-27,4859,0,0
4,HYG,2007-01-03,2026-04-27,4859,67,67
5,IEF,2007-01-03,2026-04-27,4859,0,0
6,IWM,2007-01-03,2026-04-27,4859,0,0
7,LQD,2007-01-03,2026-04-27,4859,0,0
8,QQQ,2007-01-03,2026-04-27,4859,0,0
9,SHY,2007-01-03,2026-04-27,4859,0,0


,series_id,start_date,end_date,rows,value_missing
0,BAMLC0A0CM,2023-04-28,2026-04-24,793,9
1,BAMLH0A0HYM2,2023-04-28,2026-04-24,793,8
2,DGS10,2007-01-01,2026-04-24,5040,208
3,DGS2,2007-01-01,2026-04-24,5040,208
4,DTB3,2007-01-01,2026-04-24,5040,208
5,T10Y2Y,2007-01-01,2026-04-27,5041,208
6,VIXCLS,2007-01-01,2026-04-24,5040,153


## Split Counts and Final Matrix

In [6]:
split_counts = pd.DataFrame(
    {
        "features_rows": features["split"].value_counts().sort_index(),
        "global_feature_rows": global_features["split"].value_counts().sort_index(),
        "model_matrix_rows": model_matrix["split"].value_counts().sort_index(),
    }
).fillna(0).astype(int)
display(split_counts)

obs_columns = [column for column in model_matrix.columns if column.startswith("obs_")]
return_columns = [column for column in model_matrix.columns if column.startswith("return_")]
display(
    pd.DataFrame(
        [
            {
                "model_matrix_rows": len(model_matrix),
                "obs_columns": len(obs_columns),
                "expected_observation_dim": feature_spec["observation_dim"],
                "return_columns": len(return_columns),
                "expected_return_columns": len(feature_spec["asset_order"]),
                "nan_count": int(model_matrix[obs_columns + return_columns].isna().sum().sum()),
                "inf_count": int(np.isinf(model_matrix[obs_columns + return_columns]).sum().sum()),
            }
        ]
    )
)

,features_rows,global_feature_rows,model_matrix_rows
split,,,
test,4606,329,329
train,49308,3522,3522
validation,3528,252,252


,model_matrix_rows,obs_columns,expected_observation_dim,return_columns,expected_return_columns,nan_count,inf_count
0,4103,316,316,14,14,0,0


## Feature Distribution Spot Checks

In [7]:
asset_spot_check = ["ret_1d", "vol_21d", "drawdown_63d", "rank_ret_21d"]
asset_spot_check = [column for column in asset_spot_check if column in features.columns]

raw_asset_stats = features[asset_spot_check].describe().T.add_prefix("raw_")
normalized_asset_stats = features_normalized[asset_spot_check].describe().T.add_prefix("normalized_")
display(raw_asset_stats.join(normalized_asset_stats))

global_spot_check = ["vix_z_21d", "credit_spread_z_63d", "spy_vol_21d", "spy_drawdown_63d"]
global_spot_check = [column for column in global_spot_check if column in global_features.columns]
raw_global_stats = global_features[global_spot_check].describe().T.add_prefix("raw_")
normalized_global_stats = global_features_normalized[global_spot_check].describe().T.add_prefix("normalized_")
display(raw_global_stats.join(normalized_global_stats))

,raw_count,raw_mean,raw_std,raw_min,raw_25%,raw_50%,raw_75%,raw_max,normalized_count,normalized_mean,normalized_std,normalized_min,normalized_25%,normalized_50%,normalized_75%,normalized_max
ret_1d,57442.0,0.000277,0.010272,-0.195136,-0.003676,0.000347,0.004752,0.120388,57442.0,4.169355e-03,0.992081,-3.968120,-0.410651,0.010440,0.471669,3.456569
vol_21d,57442.0,0.133862,0.094156,0.001843,0.072011,0.123331,0.173570,1.172483,57442.0,-6.281507e-03,0.988577,-1.472923,-0.694976,-0.114155,0.454434,4.927149
drawdown_63d,57442.0,-0.034217,0.042834,-0.423982,-0.049680,-0.018603,-0.003598,0.000000,57442.0,3.269497e-02,0.965230,-4.571668,-0.330138,0.389579,0.737076,0.820395
rank_ret_21d,57442.0,0.535714,0.287940,0.071429,0.285714,0.535714,0.785714,1.000000,57442.0,4.276065e-17,1.000009,-1.612452,-0.868243,0.000000,0.868243,1.612452


,raw_count,raw_mean,raw_std,raw_min,raw_25%,raw_50%,raw_75%,raw_max,normalized_count,normalized_mean,normalized_std,normalized_min,normalized_25%,normalized_50%,normalized_75%,normalized_max
vix_z_21d,4103.0,-0.033078,1.224336,-2.814737,-0.970694,-0.311662,0.781722,4.159024,4103.0,0.016846,0.997363,-1.693751,-0.751194,-0.211487,0.683927,2.957295
credit_spread_z_63d,4103.0,-0.341219,1.301045,-3.200357,-1.338552,-0.637362,0.520669,5.267095,4103.0,-0.004668,0.999962,-1.833212,-0.775271,-0.232773,0.663175,2.998482
spy_vol_21d,4103.0,0.146942,0.091032,0.034158,0.093055,0.125262,0.174590,0.936683,4103.0,-0.012621,0.986329,-1.146685,-0.612370,-0.251635,0.300866,7.225087
spy_drawdown_63d,4103.0,-0.028171,0.040227,-0.337173,-0.038030,-0.011594,-0.001682,0.000000,4103.0,0.035526,0.970884,-4.458074,-0.216404,0.443154,0.690448,0.732406


## Current Weight Slice and Return Columns

In [8]:
weight_obs_columns = obs_columns[-len(feature_spec["asset_order"]):]
weight_slice = model_matrix.loc[model_matrix.index[0], weight_obs_columns]
display(pd.DataFrame({"asset": feature_spec["asset_order"], "default_weight": weight_slice.to_numpy()}))

return_summary = model_matrix[return_columns].describe().T
display(return_summary[["mean", "std", "min", "max"]])

,asset,default_weight
0,SPY,0.071429
1,QQQ,0.071429
2,IWM,0.071429
3,EFA,0.071429
4,EEM,0.071429
5,TLT,0.071429
6,IEF,0.071429
7,SHY,0.071429
8,LQD,0.071429
9,HYG,0.071429


,mean,std,min,max
return_spy_1d,0.000524,0.010831,-0.115886,0.099863
return_qqq_1d,0.000687,0.012996,-0.127592,0.113356
return_iwm_1d,0.000417,0.014071,-0.142335,0.087545
return_efa_1d,0.000265,0.011634,-0.116424,0.081332
return_eem_1d,0.000187,0.013530,-0.133294,0.077451
return_tlt_1d,0.000105,0.009454,-0.069010,0.072503
return_ief_1d,0.000106,0.004158,-0.025392,0.026073
return_shy_1d,0.000055,0.000842,-0.005101,0.009925
return_lqd_1d,0.000156,0.004800,-0.051325,0.071314
return_hyg_1d,0.000207,0.005212,-0.056534,0.063406


## Completion Check

If all cells ran, the existing Phase 1 artifacts loaded successfully and the final matrix remained clean. Automated pass/fail validation remains in `make validate`.